Frequency range to detect: 50-500 Hz  
Up to C5 ~ 500 Hz

In [ ]:
:dep hound
use hound;

// let input_file = "./inputs/vocadito_28.wav";
let input_file = "./inputs/vocadito_38.wav";
let mut reader = hound::WavReader::open(input_file).unwrap();
let spec = reader.spec();
println!(
    "{0} channel; {1} Hz; {2} bps",
    spec.channels, spec.sample_rate, spec.bits_per_sample
);
let sample_rate = spec.sample_rate;

// Skip first 3.5s; read 1s
let samples_in_3_5s = sample_rate / 10 * 35;
let samples_in_1s = sample_rate;
let samples = reader
    .samples::<i16>()
    .skip(samples_in_3_5s as usize)
    .take(samples_in_1s as usize)
    .map(|x| x.unwrap());
let samples: Vec<i16> = samples.collect();
// Downsample to sample rate of 5kHz (upper frequency to be detected * 10)
let desired_sample_rate = 5000;
let downsample_rate = sample_rate / desired_sample_rate;
let samples: Vec<i16> = samples
    .into_iter()
    .step_by(downsample_rate as usize)
    .collect();
let actual_sample_rate = sample_rate / downsample_rate;
actual_sample_rate


In [ ]:
:dep ../../upyin
use upyin::*;

const DIFF_SIZE: usize = 100;
let frame_size = DIFF_SIZE * 2;

let mut yin:Yin<Sample, DIFF_SIZE> = Yin::new_with_sample_rate(actual_sample_rate.Hz());
// let mut yin:Yin<f32, 128> = Yin::new();

let fixed_samples: Vec<Sample> = samples
    .iter()
    .cloned()
    .map(|x| Sample::from_num(x))
    .collect();

yin.calculate_diff(&fixed_samples[..frame_size]).unwrap();
let candidates: Candidates = yin.find_candidates();

for candidate in candidates {
    // vocadito_28 at 3.5 s - expected: 117 Hz; calculated: 117 Hz
    // vocadito_38 at 3.5 s - expected: 297 Hz; calculated: 290 Hz
    println!("{} Hz", candidate.frequency());
}


In [ ]:
:dep plotters = { version = "^0.3.6", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
use plotters::prelude::*;

evcxr_figure((640, 480), |root| {

    let diff_serie: Vec<(f32, f32)> = yin.diff_function()
        .iter()
        .cloned()
        .enumerate()
        .map(|(n, x)| (n as f32 / actual_sample_rate as f32 * 1000_f32, x.to_num()))
        .collect();

    let candidates: Candidates = yin.find_candidates();
    let mut candidates_serie: Vec<Vec<(f32, f32)>> = vec![];
    for candidate in candidates {
        let x = candidate.frequency() as f32;
        let line: Vec<(f32, f32)> = vec![(1000.0 / x, 0.0), (1000.0 / x, 3.0)];
        candidates_serie.push(line);
    }

    let mut chart = ChartBuilder::on(&root)
        .caption("Diff function output", ("Arial", 20).into_font())
        .x_label_area_size(30)
        .y_label_area_size(30)
        .build_cartesian_2d(0_f32..(DIFF_SIZE as f32 /(actual_sample_rate as f32) * 1000_f32), 0_f32..3_f32)?;
    
    chart.configure_mesh()
        .y_labels(10)
        .light_line_style(&TRANSPARENT)
        .disable_x_mesh()
        .x_desc("τ, ms")
        .draw()?;

    chart.draw_series(LineSeries::new(diff_serie, &RED)
        .point_size(1))?;
    for line in candidates_serie {
        chart.draw_series(LineSeries::new(line, &BLUE)
            .point_size(0))?;
    } 
    
    Ok(())
})

In [ ]:
:dep plotters = { version = "^0.3.6", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
use plotters::prelude::*;

evcxr_figure((640, 480), |root| {

    let in_serie: Vec<(f32, f32)> = samples
        .iter()
        .cloned()
        .take(DIFF_SIZE*2)
        .enumerate()
        .map(|(n, x)| (n as f32 / actual_sample_rate as f32 * 1000_f32, x as f32))
        .collect();

    let mut chart = ChartBuilder::on(&root)
        .caption("Input samples", ("Arial", 20).into_font())
        .x_label_area_size(50)
        .y_label_area_size(50)
        .build_cartesian_2d(0_f32..(DIFF_SIZE as f32 * 2.0 /(actual_sample_rate as f32) * 1000_f32), -18000_f32..18000_f32)?;
    
    chart.configure_mesh()
        .y_labels(10)
        .light_line_style(&TRANSPARENT)
        .disable_x_mesh()
        .x_desc("t, ms")
        .draw()?;

    chart.draw_series(LineSeries::new(in_serie, &RED)
        .point_size(1))?;
    
    Ok(())
})

In [ ]:
:dep ../../upyin
use upyin::*;
use std::f32::consts::PI;

let sample_rate = 5000;
let sin_freq = 297.0;

let sin: Vec<f32> = (0..200).map(|x| (2.0 * PI * sin_freq * x as f32 / sample_rate as f32).sin()).collect();

// let mut yin:Yin<f32, 100> = Yin::new_with_sample_rate(sample_rate.Hz());
let mut yin:Yin<Sample, 100> = Yin::new_with_sample_rate(sample_rate.Hz());

let sin: Vec<Sample> = sin
    .iter()
    .cloned()
    .map(|x| Sample::from_num(x))
    .collect();

yin.calculate_diff(&sin).unwrap();
let candidates: Candidates = yin.find_candidates();

for candidate in candidates {
    println!("{} Hz", candidate.frequency());
}
